In [1]:
import pandas as pd
import sqlite3

In [2]:
connection = sqlite3.connect("./db/vendas.db")
crsr = connection.cursor()
print("Connected to the database")

Connected to the database


In [5]:
sql_query = 'SELECT * FROM vendas'
df = pd.read_sql_query(sql_query, connection)
print(df.head(10))

   id_venda vendedor regiao produto  data_venda  valor
0         1      Ana    Sul       A  2024-01-01    100
1         2      Ana    Sul       B  2024-01-02    200
2         3      Ana    Sul       A  2024-01-03    150
3         4    Bruno  Norte       A  2024-01-01    300
4         5    Bruno  Norte       B  2024-01-03    100
5         6    Bruno  Norte       A  2024-01-04    200
6         7    Carla    Sul       A  2024-01-02    250
7         8    Carla    Sul       B  2024-01-03    300
8         9    Carla    Sul       A  2024-01-05    100
9        10      Ana    Sul       B  2024-01-06    400


In [ ]:
'''1. Liste todas as vendas com um número sequencial usando ROW_NUMBER() ordenado por valor (do maior para o menor).'''
sql_query = """ SELECT vendedor, regiao, produto, data_venda, valor, ROW_NUMBER() OVER (ORDER BY valor DESC) AS row_num FROM vendas """

df_all_vendas = pd.read_sql_query(sql_query, connection)
print(df_all_vendas.head(10))

  vendedor regiao produto  data_venda  valor  rank
0      Ana    Sul       B  2024-01-06    400     1
1    Bruno  Norte       A  2024-01-01    300     2
2    Carla    Sul       B  2024-01-03    300     3
3    Carla    Sul       A  2024-01-02    250     4
4      Ana    Sul       B  2024-01-02    200     5
5    Bruno  Norte       A  2024-01-04    200     6
6      Ana    Sul       A  2024-01-03    150     7
7      Ana    Sul       A  2024-01-01    100     8
8    Bruno  Norte       B  2024-01-03    100     9
9    Carla    Sul       A  2024-01-05    100    10


In [ ]:
'''2. Gere um ranking das vendas usando RANK() baseado no valor (maior valor = rank 1).'''
sql_query = """ SELECT vendedor, regiao, produto, data_venda, valor, RANK() OVER (ORDER BY valor DESC) AS rank FROM vendas """

df_rank_vendas = pd.read_sql_query(sql_query, connection)
print(df_rank_vendas.head(10))

  vendedor regiao produto  data_venda  valor  rank
0      Ana    Sul       B  2024-01-06    400     1
1    Bruno  Norte       A  2024-01-01    300     2
2    Carla    Sul       B  2024-01-03    300     2
3    Carla    Sul       A  2024-01-02    250     4
4      Ana    Sul       B  2024-01-02    200     5
5    Bruno  Norte       A  2024-01-04    200     5
6      Ana    Sul       A  2024-01-03    150     7
7      Ana    Sul       A  2024-01-01    100     8
8    Bruno  Norte       B  2024-01-03    100     8
9    Carla    Sul       A  2024-01-05    100     8


In [ ]:

'''Gere um ranking usando DENSE_RANK() baseado no valor.'''
sql_query = """ SELECT vendedor, regiao, produto, data_venda, valor, DENSE_RANK() OVER (ORDER BY valor DESC) AS rank FROM vendas """

df_dense_rank_vendas = pd.read_sql_query(sql_query, connection)
print(df_dense_rank_vendas.head(10))

  vendedor regiao produto  data_venda  valor  rank
0      Ana    Sul       B  2024-01-06    400     1
1    Bruno  Norte       A  2024-01-01    300     2
2    Carla    Sul       B  2024-01-03    300     2
3    Carla    Sul       A  2024-01-02    250     3
4      Ana    Sul       B  2024-01-02    200     4
5    Bruno  Norte       A  2024-01-04    200     4
6      Ana    Sul       A  2024-01-03    150     5
7      Ana    Sul       A  2024-01-01    100     6
8    Bruno  Norte       B  2024-01-03    100     6
9    Carla    Sul       A  2024-01-05    100     6


In [ ]:
'''4. Use ROW_NUMBER() para numerar as vendas dentro de cada região, ordenando por valor decrescente.'''
sql_query = """ SELECT vendedor, regiao, valor, ROW_NUMBER() OVER (PARTITION BY regiao ORDER BY valor DESC) AS row_num FROM vendas """

df_all_vendas_por_regiao = pd.read_sql_query(sql_query, connection)
print(df_all_vendas_por_regiao.head(10))

  vendedor regiao  valor  row_num
0    Bruno  Norte    300        1
1    Bruno  Norte    200        2
2    Bruno  Norte    100        3
3      Ana    Sul    400        1
4    Carla    Sul    300        2
5    Carla    Sul    250        3
6      Ana    Sul    200        4
7      Ana    Sul    150        5
8      Ana    Sul    100        6
9    Carla    Sul    100        7


In [ ]:

'''5. Gere um ranking (RANK()) das vendas por regiao, baseado no valor.'''
sql_query = """ SELECT vendedor, regiao, valor, RANK() OVER (PARTITION BY regiao ORDER BY valor DESC) AS rank FROM vendas """

df_rank_vendas_por_regiao = pd.read_sql_query(sql_query, connection)
print(df_rank_vendas_por_regiao.head(10))

  vendedor regiao  valor  rank
0    Bruno  Norte    300     1
1    Bruno  Norte    200     2
2    Bruno  Norte    100     3
3      Ana    Sul    400     1
4    Carla    Sul    300     2
5    Carla    Sul    250     3
6      Ana    Sul    200     4
7      Ana    Sul    150     5
8      Ana    Sul    100     6
9    Carla    Sul    100     6


In [ ]:

'''6. Gere um ranking (DENSE_RANK()) por regiao, baseado no valor.'''
sql_query = """ SELECT vendedor, regiao, valor, DENSE_RANK() OVER (PARTITION BY regiao ORDER BY valor DESC) AS dense_rank FROM vendas """

df_dense_rank_vendas_por_regiao = pd.read_sql_query(sql_query, connection)
print(df_dense_rank_vendas_por_regiao.head(10))

  vendedor regiao  valor  dense_rank
0    Bruno  Norte    300           1
1    Bruno  Norte    200           2
2    Bruno  Norte    100           3
3      Ana    Sul    400           1
4    Carla    Sul    300           2
5    Carla    Sul    250           3
6      Ana    Sul    200           4
7      Ana    Sul    150           5
8      Ana    Sul    100           6
9    Carla    Sul    100           6


In [ ]:
'''7. Liste apenas a venda mais cara de cada região usando ROW_NUMBER().'''
sql_query = """ WITH ranked_vendas AS(
SELECT vendedor, regiao, valor, ROW_NUMBER() OVER (PARTITION BY regiao ORDER BY valor DESC) AS row_num FROM vendas) SELECT * FROM ranked_vendas WHERE row_num = 1 """

df_top_venda_por_regiao = pd.read_sql_query(sql_query, connection)
print(df_top_venda_por_regiao.head(10))

  vendedor regiao  valor  row_num
0    Bruno  Norte    300        1
1      Ana    Sul    400        1


*Se houver empate no maior valor, ROW_NUMBER() vai escolher apenas 1 linha arbitrariamente.*<br>
_Se duas vendas têm valor 800 na mesma região:_
* ROW_NUMBER() → retorna só uma
* RANK() ou DENSE_RANK() → retornariam ambas

In [14]:
'''7.5 Liste todas as vendas mais caras de cada região'''
sql_query = """ WITH ranked_vendas AS(
SELECT vendedor, regiao, valor, RANK() OVER (PARTITION BY regiao ORDER BY valor DESC) AS rank_pos FROM vendas) SELECT * FROM ranked_vendas WHERE rank_pos = 1 """

df_all_top_vendas_por_regiao = pd.read_sql_query(sql_query, connection)
print(df_all_top_vendas_por_regiao.head(10))

  vendedor regiao  valor  rank_pos
0    Bruno  Norte    300         1
1      Ana    Sul    400         1


In [18]:
'''8. Liste os top 2 valores por região usando RANK().'''
sql_query = """ WITH ranked_vendas AS(
SELECT vendedor, regiao, valor, RANK() OVER (PARTITION BY regiao ORDER BY valor DESC) AS rank_pos FROM vendas) SELECT * FROM ranked_vendas WHERE rank_pos <= 2; """

df_all_sec_vendas_por_regiao = pd.read_sql_query(sql_query, connection)
print(df_all_sec_vendas_por_regiao.head(10))

  vendedor regiao  valor  rank_pos
0    Bruno  Norte    300         1
1    Bruno  Norte    200         2
2      Ana    Sul    400         1
3    Carla    Sul    300         2


In [ ]:

'''9. Liste todas as vendas onde o ranking (DENSE_RANK()) seja menor ou igual a 2 dentro de cada região.'''
sql_query = """ WITH ranked_vendas AS(
SELECT vendedor, regiao, valor, DENSE_RANK() OVER (PARTITION BY regiao ORDER BY valor DESC) AS dense_rank FROM vendas) SELECT * FROM ranked_vendas WHERE dense_rank <= 2 """

df_tops_vendas_por_regiao = pd.read_sql_query(sql_query, connection)
print(df_tops_vendas_por_regiao.head(10))

  vendedor regiao  valor  rank_pos
0    Bruno  Norte    300         1
1    Bruno  Norte    200         2
2      Ana    Sul    400         1
3    Carla    Sul    300         2
